<a href="https://colab.research.google.com/github/yogitapal11jan/UH_Thesis_Checker/blob/main/BindCraft_IgG_Fc_Chain_A_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BindCraft: Protein binder design

<img src="https://github.com/martinpacesa/BindCraft/blob/main/pipeline.png?raw=true">

Simple binder design pipeline using AlphaFold2 backpropagation, MPNN, and PyRosetta. Select your target and let the script do the rest of the work and finish once you have enough designs to order!

The designs will be saved on your Google Drive under BindCraft/[design_name]/ and you can continue running the design pipeline if the session times out and it will continue adding new designs.

In [1]:
!nvidia-smi

Sun May 17 03:56:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -q pyrosetta-installer

import pyrosetta_installer
pyrosetta_installer.install_pyrosetta(type="MinSizeRel", silent=False)

import pyrosetta
print("PyRosetta imported successfully")

Installing PyRosetta:
 os: ubuntu
 type: MinSizeRel
 Rosetta C++ extras: 
 mirror: https://west.rosettacommons.org/pyrosetta/release/release
 extra packages: numpy

PyRosetta wheel url: https://:@west.rosettacommons.org/pyrosetta/release/release/PyRosetta4.MinSizeRel.python312.ubuntu.wheel/pyrosetta-2026.19+release.36534127fc-cp312-cp312-linux_x86_64.whl
PyRosetta imported successfully


In [13]:
# Create chain-A-only PDB for new BindCraft run

import os, glob

# Existing cleaned Fc AB dimer file from your previous run
source_candidates = [
    "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/target/5JII_Fc_AB_renumbered.pdb",
    "/content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/target/5JII_Fc_AB_renumbered.pdb"
]

source_pdb = None

for p in source_candidates:
    if os.path.exists(p):
        source_pdb = p
        break

# Backup search if exact path is not found
if source_pdb is None:
    hits = glob.glob("/content/drive/**/5JII*AB*renumbered*.pdb", recursive=True)
    if hits:
        source_pdb = hits[0]

if source_pdb is None:
    raise FileNotFoundError("Could not find the cleaned AB renumbered PDB. Please check your previous target folder.")

print("Using source PDB:")
print(source_pdb)

# New target folder and output file
new_target_dir = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CHAIN_A_Default/target"
os.makedirs(new_target_dir, exist_ok=True)

chainA_pdb = os.path.join(new_target_dir, "5JII_Fc_chainA_only.pdb")

# Write only chain A ATOM records
atom_count = 0
residues = set()

with open(source_pdb, "r") as fin, open(chainA_pdb, "w") as fout:
    for line in fin:
        if line.startswith("ATOM") and line[21] == "A":
            fout.write(line)
            atom_count += 1
            residues.add(line[22:26].strip())
    fout.write("TER\nEND\n")

print("\nCreated chain-A-only target PDB:")
print(chainA_pdb)
print("File exists:", os.path.exists(chainA_pdb))
print("Atom count:", atom_count)
print("Residue count:", len(residues))

if atom_count == 0:
    raise ValueError("No chain A atoms were found. Check whether the source PDB chain ID is really A.")

Using source PDB:
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/target/5JII_Fc_AB_renumbered.pdb

Created chain-A-only target PDB:
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CHAIN_A_Default/target/5JII_Fc_chainA_only.pdb
File exists: True
Atom count: 3282
Residue count: 207


In [14]:
#@title Installation
%%time
import os, time, gc, io
import contextlib
import json
from datetime import datetime
from ipywidgets import HTML, VBox
from IPython.display import display

if not os.path.isfile("bindcraft/params/done.txt"):
  print("Installing required BindCraft components")

  print("Pulling BindCraft code from Github")
  os.makedirs('/content/bindcraft/', exist_ok=True)
  !git clone https://github.com/martinpacesa/BindCraft /content/bindcraft/
  os.system("chmod +x /content/bindcraft/functions/dssp")
  os.system("chmod +x /content/bindcraft/functions/DAlphaBall.gcc")

  print("Installing ColabDesign")
  os.system("(mkdir bindcraft/params; apt-get install aria2 -qq; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C bindcraft/params; touch bindcraft/params/done.txt )&")
  os.system("pip install git+https://github.com/sokrypton/ColabDesign.git")
  # for debugging purposes
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

  print("Installing PyRosetta")
  os.system("pip install pyrosetta_installer")
  with contextlib.redirect_stdout(io.StringIO()):
    import pyrosetta_installer
    pyrosetta_installer.install_pyrosetta(serialization=True)

  # download params
  if not os.path.isfile("bindcraft/params/done.txt"):
    print("downloading AlphaFold params")
    while not os.path.isfile("bindcraft/params/done.txt"):
      time.sleep(5)

  print("BindCraft installation is finished, ready to run!")
else:
  print("BindCraft components already installed, ready to run!")

BindCraft components already installed, ready to run!
CPU times: user 98 µs, sys: 5 µs, total: 103 µs
Wall time: 100 µs


In [15]:
#@title Mount your Google Drive to save design results
from google.colab import drive
drive.mount('/content/drive')
currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Google drive mounted at: {currenttime}")

bindcraft_google_drive = '/content/drive/My Drive/BindCraft/'
os.makedirs(bindcraft_google_drive, exist_ok=True)
print("BindCraft folder successfully created in your drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google drive mounted at: 2026-05-17 04:07:15
BindCraft folder successfully created in your drive!


In [16]:
#@title Binder design settings
# @markdown ---
# @markdown Enter path where to save your designs. We recommend to save on Google drive so that you can continue generating at any time.
design_path = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CHAIN_A_Default/A_run1/" # @param {"type":"string","placeholder":"/content/drive/MyDrive/BindCraft/PDL1/"}

# @markdown Enter the name that should be prefixed to your binders (generally target name).
binder_name = "IgG_Fc_5JII_chainA_default" # @param {"type":"string","placeholder":"PDL1"}

# @markdown The path to the .pdb structure of your target. Can be an experimental or AlphaFold2 structure. We recommend trimming the structure to as small as needed, as the whole selected chains will be backpropagated through the network and can significantly increase running times.
starting_pdb = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CHAIN_A_Default/target/5JII_Fc_chainA_only.pdb" # @param {"type":"string","placeholder":"/content/bindcraft/example/PDL1.pdb"}

# @markdown Which chains of your PDB to target? Can be one or multiple, in a comma-separated format. Other chains will be ignored during design.
chains = "A" # @param {"type":"string","placeholder":"A,C"}

# @markdown What positions to target in your protein of interest? For example `1,2-10` or chain specific `A1-10,B1-20` or entire chains `A`. If left blank, an appropriate site will be selected by the pipeline.
target_hotspot_residues = "" # @param {"type":"string","placeholder":""}

# @markdown What is the minimum and maximum size of binders you want to design? Pipeline will randomly sample different sizes between these values.
lengths = "50,80" # @param {"type":"string","placeholder":"70,150"}

# @markdown How many binder designs passing filters do you require?
number_of_final_designs = 3 # @param {"type":"integer","placeholder":"100"}
# @markdown ---
# @markdown Enter path on your Google drive (/content/drive/MyDrive/BindCraft/[binder_name].json) to previous target settings to continue design campaign. If left empty, it will use the settings above and generate a new settings json in your design output folder.
load_previous_target_settings = "" # @param {"type":"string","placeholder":""}
# @markdown ---

if load_previous_target_settings:
    target_settings_path = load_previous_target_settings
else:
    lengths = [int(x.strip()) for x in lengths.split(',') if len(lengths.split(',')) == 2]

    if len(lengths) != 2:
        raise ValueError("Incorrect specification of binder lengths.")

    settings = {
        "design_path": design_path,
        "binder_name": binder_name,
        "starting_pdb": starting_pdb,
        "chains": chains,
        "target_hotspot_residues": target_hotspot_residues,
        "lengths": lengths,
        "number_of_final_designs": number_of_final_designs
    }

    target_settings_path = os.path.join(design_path, binder_name+".json")
    os.makedirs(design_path, exist_ok=True)

    with open(target_settings_path, 'w') as f:
        json.dump(settings, f, indent=4)

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Binder design settings updated at: {currenttime}")
print(f"New .json file with target settings has been generated in: {target_settings_path}")

Binder design settings updated at: 2026-05-17 04:07:18
New .json file with target settings has been generated in: /content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CHAIN_A_Default/A_run1/IgG_Fc_5JII_chainA_default.json


In [17]:
#@title Advanced settings
# @markdown ---
# @markdown Which binder design protocol to run? Default is recommended. "Beta-sheet" promotes the design of more beta sheeted proteins, but requires more sampling. "Peptide" is optimised for helical peptide binders.
design_protocol = "Default" # @param ["Default","Beta-sheet","Peptide"]
# @markdown What prediction protocol to use?. "Default" performs single sequence prediction of the binder. "HardTarget" uses initial guess to improve complex prediction for difficult targets, but might introduce some bias.
prediction_protocol = "Default" # @param ["Default","HardTarget"]
# @markdown What interface design method to use?. "AlphaFold2" is the default, interface is generated by AlphaFold2. "MPNN" uses soluble MPNN to optimise the interface.
interface_protocol = "AlphaFold2" # @param ["AlphaFold2","MPNN"]
# @markdown What target template protocol to use? "Default" allows for limited amount flexibility. "Masked" allows for greater target flexibility on both sidechain and backbone level.
template_protocol = "Default" # @param ["Default","Masked"]
# @markdown ---

if design_protocol == "Default":
    design_protocol_tag = "default_4stage_multimer"
elif design_protocol == "Beta-sheet":
    design_protocol_tag = "betasheet_4stage_multimer"
elif design_protocol == "Peptide":
    design_protocol_tag = "peptide_3stage_multimer"
else:
    raise ValueError(f"Unsupported design protocol")

if interface_protocol == "AlphaFold2":
    interface_protocol_tag = ""
elif interface_protocol == "MPNN":
    interface_protocol_tag = "_mpnn"
else:
    raise ValueError(f"Unsupported interface protocol")

if template_protocol == "Default":
    template_protocol_tag = ""
elif template_protocol == "Masked":
    template_protocol_tag = "_flexible"
else:
    raise ValueError(f"Unsupported template protocol")

if design_protocol in ["Peptide"]:
    prediction_protocol_tag = ""
else:
    if prediction_protocol == "Default":
        prediction_protocol_tag = ""
    elif prediction_protocol == "HardTarget":
        prediction_protocol_tag = "_hardtarget"
    else:
        raise ValueError(f"Unsupported prediction protocol")

advanced_settings_path = "/content/bindcraft/settings_advanced/" + design_protocol_tag + interface_protocol_tag + template_protocol_tag + prediction_protocol_tag + ".json"

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Advanced design settings updated at: {currenttime}")

Advanced design settings updated at: 2026-05-17 04:07:22


In [18]:
#@title Filters
# @markdown ---
# @markdown Which filters for designs to use? "Default" are recommended, "Peptide" are for the design of peptide binders, "Relaxed" are more permissive but may result in fewer experimental successes, "Peptide_Relaxed" are more permissive filters for non-helical peptides, "None" is for benchmarking.
filter_option = "Default" # @param ["Default", "Peptide", "Relaxed", "Peptide_Relaxed", "None"]
# @markdown ---

if filter_option == "Default":
    filter_settings_path = "/content/bindcraft/settings_filters/default_filters.json"
elif filter_option == "Peptide":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_filters.json"
elif filter_option == "Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/relaxed_filters.json"
elif filter_option == "Peptide_Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_relaxed_filters.json"
elif filter_option == "None":
    filter_settings_path = "/content/bindcraft/settings_filters/no_filters.json"
else:
    raise ValueError(f"Unsupported filter type")

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Filter settings updated at: {currenttime}")

Filter settings updated at: 2026-05-17 04:07:25


# Everything is set, BindCraft is ready to run!

In [19]:
# @title Import functions and settings
from bindcraft.functions import *

args = {"settings":target_settings_path,
        "filters":filter_settings_path,
        "advanced":advanced_settings_path}

# Check if JAX-capable GPU is available, otherwise exit
check_jax_gpu()

# perform checks of input setting files
settings_path, filters_path, advanced_path = (args["settings"], args["filters"], args["advanced"])

### load settings from JSON
target_settings, advanced_settings, filters = load_json_settings(settings_path, filters_path, advanced_path)

settings_file = os.path.basename(settings_path).split('.')[0]
filters_file = os.path.basename(filters_path).split('.')[0]
advanced_file = os.path.basename(advanced_path).split('.')[0]

### load AF2 model settings
design_models, prediction_models, multimer_validation = load_af2_models(advanced_settings["use_multimer_design"])

### perform checks on advanced_settings
bindcraft_folder = "colab"
advanced_settings = perform_advanced_settings_check(advanced_settings, bindcraft_folder)

### generate directories, design path names can be found within the function
design_paths = generate_directories(target_settings["design_path"])

### generate dataframes
trajectory_labels, design_labels, final_labels = generate_dataframe_labels()

trajectory_csv = os.path.join(target_settings["design_path"], 'trajectory_stats.csv')
mpnn_csv = os.path.join(target_settings["design_path"], 'mpnn_design_stats.csv')
final_csv = os.path.join(target_settings["design_path"], 'final_design_stats.csv')
failure_csv = os.path.join(target_settings["design_path"], 'failure_csv.csv')

create_dataframe(trajectory_csv, trajectory_labels)
create_dataframe(mpnn_csv, design_labels)
create_dataframe(final_csv, final_labels)
generate_filter_pass_csv(failure_csv, args["filters"])

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Loaded design functions and settings at: {currenttime}")

Available GPUs:
NVIDIA A100-SXM4-80GB1: gpu
Loaded design functions and settings at: 2026-05-17 04:07:27


In [20]:
#@title Initialise PyRosetta

####################################
####################################
####################################
### initialise PyRosetta
pr.init(f'-ignore_unrecognized_res -ignore_zero_occupancy -mute all -holes:dalphaball {advanced_settings["dalphaball_path"]} -corrections::beta_nov16 true -relax:default_repeats 1')

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.MinSizeRel.python312.ubuntu 2026.19+release.36534127fcb60cadc591e0f487c312a3fba4abc0 2026-05-07T11:56:27] retrieved from: http://www.pyrosetta.org


In [21]:
#@title Run BindCraft!
####################################
###################### BindCraft Run
####################################

# ==============================
# Yogi patch for multi-chain target A,B
# ==============================

import os
import time
import shutil
import numpy as np
import pandas as pd
from IPython.display import display
from ipywidgets import HTML, VBox

def chains_in_pdb(pdb_path):
    chains = []
    seen = set()
    with open(pdb_path, "r") as f:
        for line in f:
            if line.startswith("ATOM"):
                chain = line[21].strip()
                if chain and chain not in seen:
                    seen.add(chain)
                    chains.append(chain)
    return chains

def clean_chain_list(chain_string):
    return [c.strip() for c in str(chain_string).split(",") if c.strip()]

def get_binder_chain_for_target_complex(complex_pdb, target_chains_string):
    """
    Detect binder chain from generated complex PDB.
    For target chains A,B, the binder is usually C.
    If only A,B are found in the generated PDB, fall back to the last chain.
    """
    target_chains = clean_chain_list(target_chains_string)
    pdb_chains = chains_in_pdb(complex_pdb)

    non_target_chains = [c for c in pdb_chains if c not in target_chains]

    if len(non_target_chains) > 0:
        binder_chain = non_target_chains[-1]
    elif len(pdb_chains) > 1:
        binder_chain = pdb_chains[-1]
    else:
        binder_chain = "C" if "B" in target_chains else "B"

    print("Target chains:", target_chains)
    print("Chains found in complex:", pdb_chains)
    print("Using binder chain:", binder_chain)

    return binder_chain

def safe_target_rmsd(model_pdb, reference_pdb, target_chains_string, label="Target_RMSD"):
    try:
        return target_pdb_rmsd(model_pdb, reference_pdb, target_chains_string)
    except Exception as e:
        print(f"Warning: {label} failed; continuing design.")
        print(e)
        return "NA"

def safe_unaligned_rmsd(reference_pdb, align_pdb, reference_chain, align_chain, label="RMSD"):
    try:
        return unaligned_rmsd(reference_pdb, align_pdb, reference_chain, align_chain)
    except Exception as e:
        print(f"Warning: {label} failed; continuing design.")
        print(e)
        return "NA"


# Colab-specific: live displays
num_sampled_trajectories = len(pd.read_csv(trajectory_csv))
num_accepted_designs = len(pd.read_csv(final_csv))

sampled_trajectories_label = HTML(
    value=f"<h3 style='color: #1f77b4;'>Sampled Trajectories: <span style='color: #1f77b4;'>{num_sampled_trajectories}</span></h3>"
)

accepted_designs_label = HTML(
    value=f"<h3 style='color: #2ca02c;'>Accepted Designs: <span style='color: #2ca02c;'>{num_accepted_designs}</span></h3>"
)

display(VBox([sampled_trajectories_label, accepted_designs_label]))


# initialise counters
script_start_time = time.time()
trajectory_n = 1
accepted_designs = 0


### start design loop
while True:

    ### check if we have the target number of binders
    final_designs_reached = check_accepted_designs(
        design_paths,
        mpnn_csv,
        final_labels,
        final_csv,
        advanced_settings,
        target_settings,
        design_labels
    )

    if final_designs_reached:
        break

    ### check if we reached maximum allowed trajectories
    max_trajectories_reached = check_n_trajectories(design_paths, advanced_settings)

    if max_trajectories_reached:
        break

    ### Initialise design
    trajectory_start_time = time.time()

    # generate random seed to vary designs
    seed = int(np.random.randint(0, high=999999, size=1, dtype=int)[0])

    # sample binder design length randomly from defined distribution
    samples = np.arange(min(target_settings["lengths"]), max(target_settings["lengths"]) + 1)
    length = np.random.choice(samples)

    # load desired helicity value to sample different secondary structure contents
    helicity_value = load_helicity(advanced_settings)

    # generate design name and check if same trajectory was already run
    design_name = target_settings["binder_name"] + "_l" + str(length) + "_s" + str(seed)

    trajectory_dirs = [
        "Trajectory",
        "Trajectory/Relaxed",
        "Trajectory/LowConfidence",
        "Trajectory/Clashing"
    ]

    trajectory_exists = any(
        os.path.exists(os.path.join(design_paths[trajectory_dir], design_name + ".pdb"))
        for trajectory_dir in trajectory_dirs
    )

    if not trajectory_exists:
        print("Starting trajectory: " + design_name)

        ### Begin binder hallucination
        trajectory = binder_hallucination(
            design_name,
            target_settings["starting_pdb"],
            target_settings["chains"],
            target_settings["target_hotspot_residues"],
            length,
            seed,
            helicity_value,
            design_models,
            advanced_settings,
            design_paths,
            failure_csv
        )

        trajectory_metrics = copy_dict(trajectory._tmp["best"]["aux"]["log"])
        trajectory_pdb = os.path.join(design_paths["Trajectory"], design_name + ".pdb")

        # round the metrics to two decimal places
        trajectory_metrics = {
            k: round(v, 2) if isinstance(v, float) else v
            for k, v in trajectory_metrics.items()
        }

        # time trajectory
        trajectory_time = time.time() - trajectory_start_time
        trajectory_time_text = (
            f"{'%d hours, %d minutes, %d seconds' % (int(trajectory_time // 3600), int((trajectory_time % 3600) // 60), int(trajectory_time % 60))}"
        )

        print("Starting trajectory took: " + trajectory_time_text)
        print("")

        # Proceed if there is no trajectory termination signal
        if trajectory.aux["log"]["terminate"] == "":

            # Relax binder to calculate statistics
            trajectory_relaxed = os.path.join(design_paths["Trajectory/Relaxed"], design_name + ".pdb")
            pr_relax(trajectory_pdb, trajectory_relaxed)

            # define binder chain safely for single-chain or multi-chain target
            binder_chain = get_binder_chain_for_target_complex(
                trajectory_pdb,
                target_settings["chains"]
            )

            # Calculate clashes before and after relaxation
            num_clashes_trajectory = calculate_clash_score(trajectory_pdb)
            num_clashes_relaxed = calculate_clash_score(trajectory_relaxed)

            # secondary structure content of starting trajectory binder and interface
            trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_i_plddt, trajectory_ss_plddt = calc_ss_percentage(
                trajectory_pdb,
                advanced_settings,
                binder_chain
            )

            # analyze interface scores for relaxed af2 trajectory
            trajectory_interface_scores, trajectory_interface_AA, trajectory_interface_residues = score_interface(
                trajectory_relaxed,
                binder_chain
            )

            # starting binder sequence
            trajectory_sequence = trajectory.get_seq(get_best=True)[0]

            # analyze sequence
            traj_seq_notes = validate_design_sequence(
                trajectory_sequence,
                num_clashes_relaxed,
                advanced_settings
            )

            # target structure RMSD compared to input PDB
            trajectory_target_rmsd = safe_target_rmsd(
                trajectory_pdb,
                target_settings["starting_pdb"],
                target_settings["chains"],
                label="Trajectory target RMSD"
            )

            # save trajectory statistics into CSV
            trajectory_data = [
                design_name,
                advanced_settings["design_algorithm"],
                length,
                seed,
                helicity_value,
                target_settings["target_hotspot_residues"],
                trajectory_sequence,
                trajectory_interface_residues,
                trajectory_metrics["plddt"],
                trajectory_metrics["ptm"],
                trajectory_metrics["i_ptm"],
                trajectory_metrics["pae"],
                trajectory_metrics["i_pae"],
                trajectory_i_plddt,
                trajectory_ss_plddt,
                num_clashes_trajectory,
                num_clashes_relaxed,
                trajectory_interface_scores["binder_score"],
                trajectory_interface_scores["surface_hydrophobicity"],
                trajectory_interface_scores["interface_sc"],
                trajectory_interface_scores["interface_packstat"],
                trajectory_interface_scores["interface_dG"],
                trajectory_interface_scores["interface_dSASA"],
                trajectory_interface_scores["interface_dG_SASA_ratio"],
                trajectory_interface_scores["interface_fraction"],
                trajectory_interface_scores["interface_hydrophobicity"],
                trajectory_interface_scores["interface_nres"],
                trajectory_interface_scores["interface_interface_hbonds"],
                trajectory_interface_scores["interface_hbond_percentage"],
                trajectory_interface_scores["interface_delta_unsat_hbonds"],
                trajectory_interface_scores["interface_delta_unsat_hbonds_percentage"],
                trajectory_alpha_interface,
                trajectory_beta_interface,
                trajectory_loops_interface,
                trajectory_alpha,
                trajectory_beta,
                trajectory_loops,
                trajectory_interface_AA,
                trajectory_target_rmsd,
                trajectory_time_text,
                traj_seq_notes,
                settings_file,
                filters_file,
                advanced_file
            ]

            insert_data(trajectory_csv, trajectory_data)

            if advanced_settings["enable_mpnn"]:

                # initialise MPNN counters
                mpnn_n = 1
                accepted_mpnn = 0
                mpnn_dict = {}
                design_start_time = time.time()

                ### MPNN redesign of starting binder
                mpnn_trajectories = mpnn_gen_sequence(
                    trajectory_pdb,
                    binder_chain,
                    trajectory_interface_residues,
                    advanced_settings
                )

                existing_mpnn_sequences = set(
                    pd.read_csv(mpnn_csv, usecols=["Sequence"])["Sequence"].values
                )

                # create set of MPNN sequences with allowed amino acid composition
                restricted_AAs = (
                    set(aa.strip().upper() for aa in advanced_settings["omit_AAs"].split(","))
                    if advanced_settings["force_reject_AA"]
                    else set()
                )

                mpnn_sequences = sorted({
                    mpnn_trajectories["seq"][n][-length:]: {
                        "seq": mpnn_trajectories["seq"][n][-length:],
                        "score": mpnn_trajectories["score"][n],
                        "seqid": mpnn_trajectories["seqid"][n]
                    }
                    for n in range(advanced_settings["num_seqs"])
                    if (
                        not restricted_AAs
                        or not any(
                            aa in mpnn_trajectories["seq"][n][-length:].upper()
                            for aa in restricted_AAs
                        )
                    )
                    and mpnn_trajectories["seq"][n][-length:] not in existing_mpnn_sequences
                }.values(), key=lambda x: x["score"])

                del existing_mpnn_sequences

                # check whether any sequences are left
                if mpnn_sequences:

                    # add optimisation for increasing recycles if trajectory is beta sheeted
                    if advanced_settings["optimise_beta"] and float(trajectory_beta) > 15:
                        advanced_settings["num_recycles_validation"] = advanced_settings["optimise_beta_recycles_valid"]

                    ### Compile prediction models once for faster prediction of MPNN sequences
                    clear_mem()

                    # compile complex prediction model
                    complex_prediction_model = mk_afdesign_model(
                        protocol="binder",
                        num_recycles=advanced_settings["num_recycles_validation"],
                        data_dir=advanced_settings["af_params_dir"],
                        use_multimer=multimer_validation
                    )

                    complex_prediction_model.prep_inputs(
                        pdb_filename=target_settings["starting_pdb"],
                        chain=target_settings["chains"],
                        binder_len=length,
                        rm_target_seq=advanced_settings["rm_template_seq_predict"],
                        rm_target_sc=advanced_settings["rm_template_sc_predict"]
                    )

                    # compile binder monomer prediction model
                    binder_prediction_model = mk_afdesign_model(
                        protocol="hallucination",
                        use_templates=False,
                        initial_guess=False,
                        use_initial_atom_pos=False,
                        num_recycles=advanced_settings["num_recycles_validation"],
                        data_dir=advanced_settings["af_params_dir"],
                        use_multimer=multimer_validation
                    )

                    binder_prediction_model.prep_inputs(length=length)

                    # iterate over designed sequences
                    for mpnn_sequence in mpnn_sequences:

                        mpnn_time = time.time()

                        # generate mpnn design name numbering
                        mpnn_design_name = design_name + "_mpnn" + str(mpnn_n)
                        mpnn_score = round(mpnn_sequence["score"], 2)
                        mpnn_seqid = round(mpnn_sequence["seqid"], 2)

                        # add design to dictionary
                        mpnn_dict[mpnn_design_name] = {
                            "seq": mpnn_sequence["seq"],
                            "score": mpnn_score,
                            "seqid": mpnn_seqid
                        }

                        # save fasta sequence
                        if advanced_settings["save_mpnn_fasta"] is True:
                            save_fasta(mpnn_design_name, mpnn_sequence["seq"], design_paths)

                        ### Predict mpnn redesigned binder complex using masked templates
                        mpnn_complex_statistics, pass_af2_filters = predict_binder_complex(
                            complex_prediction_model,
                            mpnn_sequence["seq"],
                            mpnn_design_name,
                            target_settings["starting_pdb"],
                            target_settings["chains"],
                            length,
                            trajectory_pdb,
                            prediction_models,
                            advanced_settings,
                            filters,
                            design_paths,
                            failure_csv
                        )

                        # if AF2 filters are not passed then skip the scoring
                        if not pass_af2_filters:
                            print(f"Base AF2 filters not passed for {mpnn_design_name}, skipping interface scoring")
                            mpnn_n += 1
                            continue

                        # calculate statistics for each model individually
                        for model_num in prediction_models:

                            mpnn_design_pdb = os.path.join(
                                design_paths["MPNN"],
                                f"{mpnn_design_name}_model{model_num + 1}.pdb"
                            )

                            mpnn_design_relaxed = os.path.join(
                                design_paths["MPNN/Relaxed"],
                                f"{mpnn_design_name}_model{model_num + 1}.pdb"
                            )

                            if os.path.exists(mpnn_design_pdb):

                                # Calculate clashes before and after relaxation
                                num_clashes_mpnn = calculate_clash_score(mpnn_design_pdb)
                                num_clashes_mpnn_relaxed = calculate_clash_score(mpnn_design_relaxed)

                                # analyze interface scores for relaxed af2 trajectory
                                mpnn_interface_scores, mpnn_interface_AA, mpnn_interface_residues = score_interface(
                                    mpnn_design_relaxed,
                                    binder_chain
                                )

                                # secondary structure content of starting trajectory binder
                                mpnn_alpha, mpnn_beta, mpnn_loops, mpnn_alpha_interface, mpnn_beta_interface, mpnn_loops_interface, mpnn_i_plddt, mpnn_ss_plddt = calc_ss_percentage(
                                    mpnn_design_pdb,
                                    advanced_settings,
                                    binder_chain
                                )

                                # unaligned RMSD calculate to determine if binder is in the designed binding site
                                rmsd_site = safe_unaligned_rmsd(
                                    trajectory_pdb,
                                    mpnn_design_pdb,
                                    binder_chain,
                                    binder_chain,
                                    label=f"{mpnn_design_name} hotspot RMSD"
                                )

                                # calculate RMSD of target compared to input PDB
                                target_rmsd = safe_target_rmsd(
                                    mpnn_design_pdb,
                                    target_settings["starting_pdb"],
                                    target_settings["chains"],
                                    label=f"{mpnn_design_name} target RMSD"
                                )

                                # add the additional statistics to the mpnn_complex_statistics dictionary
                                mpnn_complex_statistics[model_num + 1].update({
                                    "i_pLDDT": mpnn_i_plddt,
                                    "ss_pLDDT": mpnn_ss_plddt,
                                    "Unrelaxed_Clashes": num_clashes_mpnn,
                                    "Relaxed_Clashes": num_clashes_mpnn_relaxed,
                                    "Binder_Energy_Score": mpnn_interface_scores["binder_score"],
                                    "Surface_Hydrophobicity": mpnn_interface_scores["surface_hydrophobicity"],
                                    "ShapeComplementarity": mpnn_interface_scores["interface_sc"],
                                    "PackStat": mpnn_interface_scores["interface_packstat"],
                                    "dG": mpnn_interface_scores["interface_dG"],
                                    "dSASA": mpnn_interface_scores["interface_dSASA"],
                                    "dG/dSASA": mpnn_interface_scores["interface_dG_SASA_ratio"],
                                    "Interface_SASA_%": mpnn_interface_scores["interface_fraction"],
                                    "Interface_Hydrophobicity": mpnn_interface_scores["interface_hydrophobicity"],
                                    "n_InterfaceResidues": mpnn_interface_scores["interface_nres"],
                                    "n_InterfaceHbonds": mpnn_interface_scores["interface_interface_hbonds"],
                                    "InterfaceHbondsPercentage": mpnn_interface_scores["interface_hbond_percentage"],
                                    "n_InterfaceUnsatHbonds": mpnn_interface_scores["interface_delta_unsat_hbonds"],
                                    "InterfaceUnsatHbondsPercentage": mpnn_interface_scores["interface_delta_unsat_hbonds_percentage"],
                                    "InterfaceAAs": mpnn_interface_AA,
                                    "Interface_Helix%": mpnn_alpha_interface,
                                    "Interface_BetaSheet%": mpnn_beta_interface,
                                    "Interface_Loop%": mpnn_loops_interface,
                                    "Binder_Helix%": mpnn_alpha,
                                    "Binder_BetaSheet%": mpnn_beta,
                                    "Binder_Loop%": mpnn_loops,
                                    "Hotspot_RMSD": rmsd_site,
                                    "Target_RMSD": target_rmsd
                                })

                                # save space by removing unrelaxed predicted mpnn complex pdb?
                                if advanced_settings["remove_unrelaxed_complex"]:
                                    os.remove(mpnn_design_pdb)

                        # calculate complex averages
                        mpnn_complex_averages = calculate_averages(
                            mpnn_complex_statistics,
                            handle_aa=True
                        )

                        ### Predict binder alone in single sequence mode
                        binder_statistics = predict_binder_alone(
                            binder_prediction_model,
                            mpnn_sequence["seq"],
                            mpnn_design_name,
                            length,
                            trajectory_pdb,
                            binder_chain,
                            prediction_models,
                            advanced_settings,
                            design_paths
                        )

                        # extract RMSDs of binder to the original trajectory
                        for model_num in prediction_models:

                            mpnn_binder_pdb = os.path.join(
                                design_paths["MPNN/Binder"],
                                f"{mpnn_design_name}_model{model_num + 1}.pdb"
                            )

                            rmsd_binder = "NA"

                            if os.path.exists(mpnn_binder_pdb):
                                rmsd_binder = safe_unaligned_rmsd(
                                    trajectory_pdb,
                                    mpnn_binder_pdb,
                                    binder_chain,
                                    "A",
                                    label=f"{mpnn_design_name} binder RMSD"
                                )

                            # append to statistics
                            binder_statistics[model_num + 1].update({
                                "Binder_RMSD": rmsd_binder
                            })

                            # save space by removing binder monomer models?
                            if advanced_settings["remove_binder_monomer"] and os.path.exists(mpnn_binder_pdb):
                                os.remove(mpnn_binder_pdb)

                        # calculate binder averages
                        binder_averages = calculate_averages(binder_statistics)

                        # analyze sequence
                        seq_notes = validate_design_sequence(
                            mpnn_sequence["seq"],
                            mpnn_complex_averages.get("Relaxed_Clashes", None),
                            advanced_settings
                        )

                        # measure time to generate design
                        mpnn_end_time = time.time() - mpnn_time
                        elapsed_mpnn_text = (
                            f"{'%d hours, %d minutes, %d seconds' % (int(mpnn_end_time // 3600), int((mpnn_end_time % 3600) // 60), int(mpnn_end_time % 60))}"
                        )

                        # Insert statistics about MPNN design into CSV
                        model_numbers = range(1, 6)

                        statistics_labels = [
                            "pLDDT",
                            "pTM",
                            "i_pTM",
                            "pAE",
                            "i_pAE",
                            "i_pLDDT",
                            "ss_pLDDT",
                            "Unrelaxed_Clashes",
                            "Relaxed_Clashes",
                            "Binder_Energy_Score",
                            "Surface_Hydrophobicity",
                            "ShapeComplementarity",
                            "PackStat",
                            "dG",
                            "dSASA",
                            "dG/dSASA",
                            "Interface_SASA_%",
                            "Interface_Hydrophobicity",
                            "n_InterfaceResidues",
                            "n_InterfaceHbonds",
                            "InterfaceHbondsPercentage",
                            "n_InterfaceUnsatHbonds",
                            "InterfaceUnsatHbondsPercentage",
                            "Interface_Helix%",
                            "Interface_BetaSheet%",
                            "Interface_Loop%",
                            "Binder_Helix%",
                            "Binder_BetaSheet%",
                            "Binder_Loop%",
                            "InterfaceAAs",
                            "Hotspot_RMSD",
                            "Target_RMSD"
                        ]

                        # Initialize mpnn_data with the non-statistical data
                        mpnn_data = [
                            mpnn_design_name,
                            advanced_settings["design_algorithm"],
                            length,
                            seed,
                            helicity_value,
                            target_settings["target_hotspot_residues"],
                            mpnn_sequence["seq"],
                            mpnn_interface_residues,
                            mpnn_score,
                            mpnn_seqid
                        ]

                        # Add the statistical data for mpnn_complex
                        for label in statistics_labels:
                            mpnn_data.append(mpnn_complex_averages.get(label, None))
                            for model in model_numbers:
                                mpnn_data.append(mpnn_complex_statistics.get(model, {}).get(label, None))

                        # Add the statistical data for binder
                        for label in ["pLDDT", "pTM", "pAE", "Binder_RMSD"]:
                            mpnn_data.append(binder_averages.get(label, None))
                            for model in model_numbers:
                                mpnn_data.append(binder_statistics.get(model, {}).get(label, None))

                        # Add the remaining non-statistical data
                        mpnn_data.extend([
                            elapsed_mpnn_text,
                            seq_notes,
                            settings_file,
                            filters_file,
                            advanced_file
                        ])

                        # insert data into csv
                        insert_data(mpnn_csv, mpnn_data)

                        # find best model number by pLDDT
                        plddt_values = {
                            i: mpnn_data[i]
                            for i in range(11, 15)
                            if mpnn_data[i] is not None
                        }

                        highest_plddt_key = int(max(plddt_values, key=plddt_values.get))
                        best_model_number = highest_plddt_key - 10

                        best_model_pdb = os.path.join(
                            design_paths["MPNN/Relaxed"],
                            f"{mpnn_design_name}_model{best_model_number}.pdb"
                        )

                        # run design data against filter thresholds
                        filter_conditions = check_filters(mpnn_data, design_labels, filters)

                        if filter_conditions == True:
                            print(mpnn_design_name + " passed all filters")
                            accepted_mpnn += 1
                            accepted_designs += 1

                            # copy designs to accepted folder
                            shutil.copy(best_model_pdb, design_paths["Accepted"])

                            # insert data into final csv
                            final_data = [""] + mpnn_data
                            insert_data(final_csv, final_data)

                            # copy animation from accepted trajectory
                            if advanced_settings["save_design_animations"]:
                                accepted_animation = os.path.join(
                                    design_paths["Accepted/Animation"],
                                    f"{design_name}.html"
                                )

                                if not os.path.exists(accepted_animation):
                                    shutil.copy(
                                        os.path.join(design_paths["Trajectory/Animation"], f"{design_name}.html"),
                                        accepted_animation
                                    )

                            # copy plots of accepted trajectory
                            plot_files = os.listdir(design_paths["Trajectory/Plots"])
                            plots_to_copy = [
                                f for f in plot_files
                                if f.startswith(design_name) and f.endswith(".png")
                            ]

                            for accepted_plot in plots_to_copy:
                                source_plot = os.path.join(design_paths["Trajectory/Plots"], accepted_plot)
                                target_plot = os.path.join(design_paths["Accepted/Plots"], accepted_plot)

                                if not os.path.exists(target_plot):
                                    shutil.copy(source_plot, target_plot)

                        else:
                            print(f"Unmet filter conditions for {mpnn_design_name}")

                            failure_df = pd.read_csv(failure_csv)
                            special_prefixes = ("Average_", "1_", "2_", "3_", "4_", "5_")
                            incremented_columns = set()

                            for column in filter_conditions:
                                base_column = column

                                for prefix in special_prefixes:
                                    if column.startswith(prefix):
                                        base_column = column.split("_", 1)[1]

                                if base_column not in incremented_columns:
                                    failure_df[base_column] = failure_df[base_column] + 1
                                    incremented_columns.add(base_column)

                            failure_df.to_csv(failure_csv, index=False)
                            shutil.copy(best_model_pdb, design_paths["Rejected"])

                        # increase MPNN design number
                        mpnn_n += 1

                        # if enough mpnn sequences of the same trajectory pass filters then stop
                        if accepted_mpnn >= advanced_settings["max_mpnn_sequences"]:
                            break

                    if accepted_mpnn >= 1:
                        print("Found " + str(accepted_mpnn) + " MPNN designs passing filters")
                    else:
                        print("No accepted MPNN designs found for this trajectory.")

                else:
                    print("Duplicate MPNN designs sampled with different trajectory, skipping current trajectory optimisation")

                # save space by removing unrelaxed design trajectory PDB
                if advanced_settings["remove_unrelaxed_trajectory"]:
                    os.remove(trajectory_pdb)

                # measure time it took to generate designs for one trajectory
                design_time = time.time() - design_start_time
                design_time_text = (
                    f"{'%d hours, %d minutes, %d seconds' % (int(design_time // 3600), int((design_time % 3600) // 60), int(design_time % 60))}"
                )

                print("Design and validation of trajectory " + design_name + " took: " + design_time_text)

            # analyse rejection rate
            if trajectory_n >= advanced_settings["start_monitoring"] and advanced_settings["enable_rejection_check"]:

                acceptance = accepted_designs / trajectory_n

                if not acceptance >= advanced_settings["acceptance_rate"]:
                    print("The ratio of successful designs is lower than defined acceptance rate! Consider changing your design settings!")
                    print("Script execution stopping...")
                    break

        # increase trajectory number
        trajectory_n += 1

        # Colab-specific: update counters
        num_sampled_trajectories = len(pd.read_csv(trajectory_csv))
        num_accepted_designs = len(pd.read_csv(final_csv))

        sampled_trajectories_label.value = f"Sampled trajectories: {num_sampled_trajectories}"
        accepted_designs_label.value = f"Accepted designs: {num_accepted_designs}"


### Script finished
elapsed_time = time.time() - script_start_time
elapsed_text = (
    f"{'%d hours, %d minutes, %d seconds' % (int(elapsed_time // 3600), int((elapsed_time % 3600) // 60), int(elapsed_time % 60))}"
)

print("Finished all designs. Script execution for " + str(trajectory_n) + " trajectories took: " + elapsed_text)

Starting trajectory: IgG_Fc_5JII_chainA_default_l53_s305933
Stage 1: Test Logits


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os, glob

search_roots = [
    "/content/drive/MyDrive",
    "/content/drive/My Drive"
]

for root in search_roots:
    print("\nSearching root:", root)
    if os.path.exists(root):
        print("BindCraft folders:")
        for path in glob.glob(root + "/**/BindCraft*", recursive=True):
            print(path)

        print("\n5JII / IgG / Fc files:")
        for pattern in ["**/*5JII*", "**/*IgG*", "**/*Fc*"]:
            for path in glob.glob(os.path.join(root, pattern), recursive=True):
                print(path)


Searching root: /content/drive/MyDrive
BindCraft folders:
/content/drive/MyDrive/BindCraft
/content/drive/MyDrive/Colab Notebooks/BindCraft_Yogita_first_test.ipynb
/content/drive/MyDrive/content/drive/MyDrive/BindCraft

5JII / IgG / Fc files:
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/target/5JII_Fc_AB_renumbered.pdb
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/target/5JII_raw.pdb
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/IgG_Fc_5JII_AB.json
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/Accepted/IgG_Fc_5JII_AB_l63_s343113_mpnn1_model2.pdb
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/Accepted/IgG_Fc_5JII_AB_l72_s140142_mpnn1_model2.pdb
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/Accepted/IgG_Fc_5JII_AB_l72_s140142_mpnn4_model2.pdb
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/Accepted/Ranked/1_IgG_Fc_5JII_AB_l72_s140142_mpnn1_model2.pdb
/content/drive

In [ ]:
import os, glob
import pandas as pd

# -----------------------------
# Option 1: If you know your current run folder, put it here.
# Leave as None to auto-search.
# -----------------------------
BASE = None
# Example:
# BASE = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"
# BASE = "/content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"

# -----------------------------
# Auto-detect possible BindCraft folders
# -----------------------------
possible_roots = [
    "/content/drive/MyDrive/BindCraft",
    "/content/drive/My Drive/BindCraft",
    "/content/BindCraft",
    "/content/bindcraft",
]

def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Could not read {path}: {e}")
        return None

def check_run_folder(base):
    print("\n" + "="*80)
    print("Checking run folder:")
    print(base)
    print("="*80)

    accepted_dirs = glob.glob(os.path.join(base, "**", "Accepted"), recursive=True)
    final_csvs = glob.glob(os.path.join(base, "**", "final_design_stats.csv"), recursive=True)
    mpnn_csvs = glob.glob(os.path.join(base, "**", "mpnn_design_stats.csv"), recursive=True)
    traj_csvs = glob.glob(os.path.join(base, "**", "trajectory_stats.csv"), recursive=True)

    # Accepted PDBs
    accepted_pdbs = []
    for d in accepted_dirs:
        accepted_pdbs.extend(glob.glob(os.path.join(d, "*.pdb")))

    print(f"Accepted folders found: {len(accepted_dirs)}")
    print(f"Accepted PDB files found: {len(accepted_pdbs)}")

    if accepted_pdbs:
        print("\nAccepted binder PDBs:")
        for p in accepted_pdbs:
            print("  ", p)

    # Final design CSV
    print(f"\nfinal_design_stats.csv files found: {len(final_csvs)}")

    total_final_rows = 0
    for csv in final_csvs:
        df = safe_read_csv(csv)
        if df is not None:
            print(f"\nFinal CSV: {csv}")
            print("Rows:", len(df))
            total_final_rows += len(df)

            if len(df) > 0:
                print("\nAccepted/final designs:")
                display(df.tail(min(len(df), 20)))

    # MPNN CSV
    print(f"\nmpnn_design_stats.csv files found: {len(mpnn_csvs)}")
    for csv in mpnn_csvs:
        df = safe_read_csv(csv)
        if df is not None:
            print(f"\nMPNN CSV: {csv}")
            print("Rows:", len(df))
            if len(df) > 0:
                display(df.tail(min(len(df), 10)))

    # Trajectory CSV
    print(f"\ntrajectory_stats.csv files found: {len(traj_csvs)}")
    for csv in traj_csvs:
        df = safe_read_csv(csv)
        if df is not None:
            print(f"\nTrajectory CSV: {csv}")
            print("Rows:", len(df))
            if len(df) > 0:
                display(df.tail(min(len(df), 10)))

    print("\nSummary:")
    print("Accepted PDB count:", len(accepted_pdbs))
    print("Final design CSV total rows:", total_final_rows)

    if len(accepted_pdbs) > 0 or total_final_rows > 0:
        print("RESULT: At least one binder appears to have passed filters.")
    else:
        print("RESULT: No accepted/final binder found in this folder yet.")

# -----------------------------
# If BASE is provided, check it directly.
# Otherwise search likely BindCraft folders.
# -----------------------------
if BASE is not None:
    check_run_folder(BASE)
else:
    print("Auto-searching for BindCraft run folders...\n")

    candidate_folders = []

    for root in possible_roots:
        if os.path.exists(root):
            print("Found root:", root)

            # folders likely related to your IgG/Fc/5JII runs
            for path in glob.glob(os.path.join(root, "**"), recursive=True):
                if os.path.isdir(path):
                    name = path.lower()
                    if any(key in name for key in ["igg", "fc", "5jii", "ab_run", "chaina"]):
                        candidate_folders.append(path)

    candidate_folders = sorted(set(candidate_folders))

    print("\nCandidate folders found:", len(candidate_folders))

    if len(candidate_folders) == 0:
        print("No candidate BindCraft folders found. Check whether Google Drive is mounted.")
    else:
        # check only folders that contain CSVs or Accepted folder
        useful_folders = []
        for folder in candidate_folders:
            has_final = glob.glob(os.path.join(folder, "**", "final_design_stats.csv"), recursive=True)
            has_mpnn = glob.glob(os.path.join(folder, "**", "mpnn_design_stats.csv"), recursive=True)
            has_traj = glob.glob(os.path.join(folder, "**", "trajectory_stats.csv"), recursive=True)
            has_accepted = glob.glob(os.path.join(folder, "**", "Accepted"), recursive=True)

            if has_final or has_mpnn or has_traj or has_accepted:
                useful_folders.append(folder)

        useful_folders = sorted(set(useful_folders))

        print("Useful folders with BindCraft outputs:", len(useful_folders))

        for folder in useful_folders:
            check_run_folder(folder)

Auto-searching for BindCraft run folders...

Found root: /content/drive/MyDrive/BindCraft
Found root: /content/drive/My Drive/BindCraft
Found root: /content/bindcraft

Candidate folders found: 40
Useful folders with BindCraft outputs: 4

Checking run folder:
/content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN
Accepted folders found: 1
Accepted PDB files found: 3

Accepted binder PDBs:
   /content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/Accepted/IgG_Fc_5JII_AB_l63_s343113_mpnn1_model2.pdb
   /content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/Accepted/IgG_Fc_5JII_AB_l72_s140142_mpnn1_model2.pdb
   /content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/Accepted/IgG_Fc_5JII_AB_l72_s140142_mpnn4_model2.pdb

final_design_stats.csv files found: 1

Final CSV: /content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/final_design_stats.csv
Rows: 3

Accepted/final designs:


,Rank,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,MPNN_score,...,1_Binder_RMSD,2_Binder_RMSD,3_Binder_RMSD,4_Binder_RMSD,5_Binder_RMSD,DesignTime,Notes,TargetSettings,Filters,AdvancedSettings
0,1,IgG_Fc_5JII_AB_l72_s140142_mpnn1,4stage,72,140142,-0.3,NaN,MIEELKEKMMNMWKLSEEERYKVAMEYFELYDKIENEYKKAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B29...",0.90,...,1.18,1.38,NaN,NaN,NaN,"0 hours, 4 minutes, 25 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
1,2,IgG_Fc_5JII_AB_l72_s140142_mpnn4,4stage,72,140142,-0.3,NaN,MIEELKEEMMNMWKKSEEERYEIAMKYFELYDKIENEYKEAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B31...",0.91,...,1.03,1.70,NaN,NaN,NaN,"0 hours, 2 minutes, 47 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
2,3,IgG_Fc_5JII_AB_l63_s343113_mpnn1,4stage,63,343113,-0.3,NaN,APLPLEEQRRRAEEEALQERRDELLREINELHDKAWELMDKDKEEY...,"B18,B22,B25,B26,B28,B29,B32,B33,B35,B36,B39,B4...",1.15,...,1.47,1.44,NaN,NaN,NaN,"0 hours, 4 minutes, 2 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer



mpnn_design_stats.csv files found: 1

MPNN CSV: /content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/mpnn_design_stats.csv
Rows: 19


,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,MPNN_score,MPNN_seq_recovery,...,1_Binder_RMSD,2_Binder_RMSD,3_Binder_RMSD,4_Binder_RMSD,5_Binder_RMSD,DesignTime,Notes,TargetSettings,Filters,AdvancedSettings
9,IgG_Fc_5JII_AB_l67_s307316_mpnn13,4stage,67,307316,-0.3,NaN,MSLAERLIEFFKKNKQEYLELNKDFVDDGFLSEEELEEMYERFLVG...,"B6,B18,B22,B24,B25,B28,B30,B31,B39,B42,B43,B45...",0.84,0.56,...,1.90,1.71,NaN,NaN,NaN,"0 hours, 3 minutes, 0 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
10,IgG_Fc_5JII_AB_l67_s307316_mpnn14,4stage,67,307316,-0.3,NaN,MSLAERLLKYFEENPEEYLNRNKDFVDDGFLSEEELEEMYKRFLVG...,"B3,B6,B18,B22,B24,B25,B28,B30,B31,B39,B42,B43,...",0.85,0.44,...,1.34,1.38,NaN,NaN,NaN,"0 hours, 2 minutes, 58 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
11,IgG_Fc_5JII_AB_l67_s307316_mpnn15,4stage,67,307316,-0.3,NaN,MSLAERLIKHFEANPQEYLDMNKDFVDDGFLSEEELKEMFERFMVG...,"B6,B18,B21,B22,B24,B25,B27,B28,B30,B31,B39,B42...",0.85,0.46,...,1.73,1.59,NaN,NaN,NaN,"0 hours, 3 minutes, 4 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
12,IgG_Fc_5JII_AB_l67_s307316_mpnn16,4stage,67,307316,-0.3,NaN,MSLAERLLKYFKENPEEYLERNKDFVDDGFLSEEELEEMYKRFLVG...,"B6,B18,B21,B22,B24,B25,B28,B30,B31,B39,B42,B43...",0.85,0.46,...,1.55,1.55,NaN,NaN,NaN,"0 hours, 3 minutes, 6 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
13,IgG_Fc_5JII_AB_l67_s307316_mpnn17,4stage,67,307316,-0.3,NaN,MSLAERLLEYFKKNPQEYLELNKDFVDDGFLSEEELEEMYERFLVG...,"B3,B18,B22,B24,B25,B28,B30,B31,B39,B42,B43,B45...",0.86,0.54,...,1.76,1.76,NaN,NaN,NaN,"0 hours, 2 minutes, 58 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
14,IgG_Fc_5JII_AB_l67_s307316_mpnn18,4stage,67,307316,-0.3,NaN,MSLAERLLEYFKKNKEDYLNENKDFVDDGFLSEEELEEMYERFLVG...,"B3,B18,B21,B22,B24,B25,B28,B30,B31,B39,B42,B43...",0.86,0.54,...,1.45,8.10,NaN,NaN,NaN,"0 hours, 3 minutes, 6 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
15,IgG_Fc_5JII_AB_l67_s307316_mpnn19,4stage,67,307316,-0.3,NaN,MSLAERLLEYFEKNEQDYLDLNKDFVDDGFLSEEELEEMYKRFLVG...,"B1,B3,B18,B21,B22,B24,B25,B28,B30,B31,B39,B42,...",0.86,0.51,...,1.62,1.68,NaN,NaN,NaN,"0 hours, 3 minutes, 1 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
16,IgG_Fc_5JII_AB_l82_s329629_mpnn20,4stage,82,329629,-0.3,NaN,MSMVDQYEMAVERALWMLSKLPESVPAVAKLRKEVEGRYEEYKKTE...,"B1,B3,B6,B7,B9,B10,B11,B13,B14,B16,B17,B20,B51...",1.03,0.46,...,6.33,4.79,NaN,NaN,NaN,"0 hours, 3 minutes, 54 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
17,IgG_Fc_5JII_AB_l72_s140142_mpnn1,4stage,72,140142,-0.3,NaN,MIEELKEKMMNMWKLSEEERYKVAMEYFELYDKIENEYKKAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B29...",0.90,0.37,...,1.18,1.38,NaN,NaN,NaN,"0 hours, 4 minutes, 25 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
18,IgG_Fc_5JII_AB_l72_s140142_mpnn4,4stage,72,140142,-0.3,NaN,MIEELKEEMMNMWKKSEEERYEIAMKYFELYDKIENEYKEAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B31...",0.91,0.31,...,1.03,1.70,NaN,NaN,NaN,"0 hours, 2 minutes, 47 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer



trajectory_stats.csv files found: 1

Trajectory CSV: /content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/trajectory_stats.csv
Rows: 57


...


mpnn_design_stats.csv files found: 1

MPNN CSV: /content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/mpnn_design_stats.csv
Rows: 19


,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,MPNN_score,MPNN_seq_recovery,...,1_Binder_RMSD,2_Binder_RMSD,3_Binder_RMSD,4_Binder_RMSD,5_Binder_RMSD,DesignTime,Notes,TargetSettings,Filters,AdvancedSettings
9,IgG_Fc_5JII_AB_l67_s307316_mpnn13,4stage,67,307316,-0.3,NaN,MSLAERLIEFFKKNKQEYLELNKDFVDDGFLSEEELEEMYERFLVG...,"B6,B18,B22,B24,B25,B28,B30,B31,B39,B42,B43,B45...",0.84,0.56,...,1.90,1.71,NaN,NaN,NaN,"0 hours, 3 minutes, 0 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
10,IgG_Fc_5JII_AB_l67_s307316_mpnn14,4stage,67,307316,-0.3,NaN,MSLAERLLKYFEENPEEYLNRNKDFVDDGFLSEEELEEMYKRFLVG...,"B3,B6,B18,B22,B24,B25,B28,B30,B31,B39,B42,B43,...",0.85,0.44,...,1.34,1.38,NaN,NaN,NaN,"0 hours, 2 minutes, 58 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
11,IgG_Fc_5JII_AB_l67_s307316_mpnn15,4stage,67,307316,-0.3,NaN,MSLAERLIKHFEANPQEYLDMNKDFVDDGFLSEEELKEMFERFMVG...,"B6,B18,B21,B22,B24,B25,B27,B28,B30,B31,B39,B42...",0.85,0.46,...,1.73,1.59,NaN,NaN,NaN,"0 hours, 3 minutes, 4 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
12,IgG_Fc_5JII_AB_l67_s307316_mpnn16,4stage,67,307316,-0.3,NaN,MSLAERLLKYFKENPEEYLERNKDFVDDGFLSEEELEEMYKRFLVG...,"B6,B18,B21,B22,B24,B25,B28,B30,B31,B39,B42,B43...",0.85,0.46,...,1.55,1.55,NaN,NaN,NaN,"0 hours, 3 minutes, 6 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
13,IgG_Fc_5JII_AB_l67_s307316_mpnn17,4stage,67,307316,-0.3,NaN,MSLAERLLEYFKKNPQEYLELNKDFVDDGFLSEEELEEMYERFLVG...,"B3,B18,B22,B24,B25,B28,B30,B31,B39,B42,B43,B45...",0.86,0.54,...,1.76,1.76,NaN,NaN,NaN,"0 hours, 2 minutes, 58 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
14,IgG_Fc_5JII_AB_l67_s307316_mpnn18,4stage,67,307316,-0.3,NaN,MSLAERLLEYFKKNKEDYLNENKDFVDDGFLSEEELEEMYERFLVG...,"B3,B18,B21,B22,B24,B25,B28,B30,B31,B39,B42,B43...",0.86,0.54,...,1.45,8.10,NaN,NaN,NaN,"0 hours, 3 minutes, 6 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
15,IgG_Fc_5JII_AB_l67_s307316_mpnn19,4stage,67,307316,-0.3,NaN,MSLAERLLEYFEKNEQDYLDLNKDFVDDGFLSEEELEEMYKRFLVG...,"B1,B3,B18,B21,B22,B24,B25,B28,B30,B31,B39,B42,...",0.86,0.51,...,1.62,1.68,NaN,NaN,NaN,"0 hours, 3 minutes, 1 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
16,IgG_Fc_5JII_AB_l82_s329629_mpnn20,4stage,82,329629,-0.3,NaN,MSMVDQYEMAVERALWMLSKLPESVPAVAKLRKEVEGRYEEYKKTE...,"B1,B3,B6,B7,B9,B10,B11,B13,B14,B16,B17,B20,B51...",1.03,0.46,...,6.33,4.79,NaN,NaN,NaN,"0 hours, 3 minutes, 54 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
17,IgG_Fc_5JII_AB_l72_s140142_mpnn1,4stage,72,140142,-0.3,NaN,MIEELKEKMMNMWKLSEEERYKVAMEYFELYDKIENEYKKAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B29...",0.90,0.37,...,1.18,1.38,NaN,NaN,NaN,"0 hours, 4 minutes, 25 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
18,IgG_Fc_5JII_AB_l72_s140142_mpnn4,4stage,72,140142,-0.3,NaN,MIEELKEEMMNMWKKSEEERYEIAMKYFELYDKIENEYKEAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B31...",0.91,0.31,...,1.03,1.70,NaN,NaN,NaN,"0 hours, 2 minutes, 47 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer



trajectory_stats.csv files found: 1

Trajectory CSV: /content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/trajectory_stats.csv
Rows: 57


,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,pLDDT,pTM,...,Binder_Helix%,Binder_BetaSheet%,Binder_Loop%,InterfaceAAs,Target_RMSD,TrajectoryTime,Notes,TargetSettings,Filters,AdvancedSettings
47,IgG_Fc_5JII_AB_l86_s114043,4stage,86,114043,-0.3,NaN,GLTPEEKVERVIRMVEKMFGKDLVVFVESDMHVIIAGHYVFIVNRE...,"B42,B50,B51,B52,B53,B55,B66,B67,B70,B71,B73,B7...",0.89,0.73,...,47.67,26.74,25.58,"{'A': 1, 'C': 0, 'D': 0, 'E': 4, 'F': 0, 'G': ...",0.91,"0 hours, 10 minutes, 22 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
48,IgG_Fc_5JII_AB_l89_s395285,4stage,89,395285,-0.3,NaN,MSIMWDSRVKTMEEWIRWMGRYSGVPTKEVRRLYEESGETFPYHPW...,"B2,B3,B5,B46,B47,B48,B49,B50,B51,B58,B61,B65,B...",0.86,0.79,...,57.30,5.62,37.08,"{'A': 0, 'C': 0, 'D': 3, 'E': 1, 'F': 1, 'G': ...",1.31,"0 hours, 8 minutes, 49 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
49,IgG_Fc_5JII_AB_l67_s307316,4stage,67,307316,-0.3,NaN,MSIASRMIEYFRGNKQDYLEQNKDFTDDGFLTQAELESMYERFMVG...,"B6,B18,B22,B24,B25,B27,B28,B30,B31,B39,B42,B43...",0.84,0.84,...,80.60,0.00,19.40,"{'A': 0, 'C': 0, 'D': 3, 'E': 1, 'F': 4, 'G': ...",1.01,"0 hours, 8 minutes, 27 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
50,IgG_Fc_5JII_AB_l69_s938435,4stage,69,938435,-0.3,NaN,MAPTPRTREHMMLDTFMDVYEKVEKKKKDNGEQRGGYVTSKEIKEE...,"B3,B4,B5,B6,B7,B9,B10,B13,B14,B16,B17,B20,B21,...",0.81,0.78,...,71.01,0.00,28.99,"{'A': 0, 'C': 0, 'D': 1, 'E': 3, 'F': 2, 'G': ...",1.29,"0 hours, 8 minutes, 28 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
51,IgG_Fc_5JII_AB_l81_s895465,4stage,81,895465,-0.3,NaN,DPMMKEMIYDTIQRRWHPHLDEVMDGYEKGYMPPEAIRRLKREVRT...,"B4,B5,B8,B9,B12,B13,B17,B20,B24,B27,B28,B60,B6...",0.81,0.77,...,86.42,0.00,13.58,"{'A': 1, 'C': 0, 'D': 1, 'E': 3, 'F': 1, 'G': ...",1.25,"0 hours, 8 minutes, 42 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
52,IgG_Fc_5JII_AB_l66_s248063,4stage,66,248063,-0.3,NaN,RKPTEAEVNEVHIKFHRSMEAHEKMMAETDPLKDKDPPRYWEEKKN...,"B5,B9,B12,B13,B15,B16,B19,B20,B22,B23,B44,B47,...",0.89,0.77,...,87.88,0.00,12.12,"{'A': 1, 'C': 0, 'D': 0, 'E': 5, 'F': 1, 'G': ...",1.17,"0 hours, 8 minutes, 29 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
53,IgG_Fc_5JII_AB_l82_s329629,4stage,82,329629,-0.3,NaN,MEMVEQYRMAVERAIWMLMKMPKDVPAVRRMTRELRGDYRRFKKTV...,"B1,B3,B6,B7,B9,B10,B11,B13,B14,B16,B17,B20,B51...",0.92,0.81,...,73.17,0.00,26.83,"{'A': 2, 'C': 0, 'D': 4, 'E': 1, 'F': 1, 'G': ...",1.05,"0 hours, 9 minutes, 12 seconds",Relaxed structure contains clashes.,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
54,IgG_Fc_5JII_AB_l53_s405039,4stage,53,405039,-0.3,NaN,MMPPAQVREMTRVLDEMIRGARWAARQSRSPGHLNKMVRDLRAIRR...,"B2,B9,B10,B12,B13,B14,B16,B17,B20,B21,B23,B24,...",0.93,0.80,...,81.13,0.00,18.87,"{'A': 3, 'C': 0, 'D': 1, 'E': 2, 'F': 0, 'G': ...",1.41,"0 hours, 8 minutes, 10 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
55,IgG_Fc_5JII_AB_l67_s916619,4stage,67,916619,-0.3,NaN,MTPAELRATLVRHVRAMIRNGPEMVMRFMYWATDEVPEEVEYYIRH...,"B1,B6,B9,B10,B13,B23,B26,B27,B28,B30,B31,B32,B...",0.86,0.74,...,77.61,0.00,22.39,"{'A': 1, 'C': 0, 'D': 1, 'E': 4, 'F': 1, 'G': ...",2.81,"0 hours, 8 minutes, 30 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
56,IgG_Fc_5JII_AB_l72_s140142,4stage,72,140142,-0.3,NaN,WVRRLEKRMMNMWKLRPRERYMVAMEYFRMYDIYENHYKNAMNDPN...,"B9,B10,B12,B13,B20,B21,B24,B25,B27,B28,B31,B32...",0.91,0.81,...,83.33,0.00,16.67,"{'A': 1, 'C': 0, 'D': 1, 'E': 1, 'F': 1, 'G': ...",2.24,"0 hours, 8 minutes, 52 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer



Summary:
Accepted PDB count: 3
Final design CSV total rows: 3
RESULT: At least one binder appears to have passed filters.


In [ ]:
import os, glob
import pandas as pd
from google.colab import files

BASE = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"
# BASE = "/content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"

final_csvs = glob.glob(os.path.join(BASE, "**", "final_design_stats.csv"), recursive=True)

print("final_design_stats.csv found:", final_csvs)

if final_csvs:
    final_csv = final_csvs[0]
    df = pd.read_csv(final_csv)
    print("Accepted binders:", len(df))
    display(df)

    output_csv = "/content/accepted_binders_final_design_stats.csv"
    df.to_csv(output_csv, index=False)

    files.download(output_csv)
else:
    print("No final_design_stats.csv found. This usually means no binder has passed filters yet, or the path is wrong.")

final_design_stats.csv found: ['/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/final_design_stats.csv']
Accepted binders: 3


,Rank,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,MPNN_score,...,1_Binder_RMSD,2_Binder_RMSD,3_Binder_RMSD,4_Binder_RMSD,5_Binder_RMSD,DesignTime,Notes,TargetSettings,Filters,AdvancedSettings
0,1,IgG_Fc_5JII_AB_l72_s140142_mpnn1,4stage,72,140142,-0.3,NaN,MIEELKEKMMNMWKLSEEERYKVAMEYFELYDKIENEYKKAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B29...",0.90,...,1.18,1.38,NaN,NaN,NaN,"0 hours, 4 minutes, 25 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
1,2,IgG_Fc_5JII_AB_l72_s140142_mpnn4,4stage,72,140142,-0.3,NaN,MIEELKEEMMNMWKKSEEERYEIAMKYFELYDKIENEYKEAMEDPN...,"B9,B10,B12,B13,B17,B20,B21,B24,B25,B27,B28,B31...",0.91,...,1.03,1.70,NaN,NaN,NaN,"0 hours, 2 minutes, 47 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer
2,3,IgG_Fc_5JII_AB_l63_s343113_mpnn1,4stage,63,343113,-0.3,NaN,APLPLEEQRRRAEEEALQERRDELLREINELHDKAWELMDKDKEEY...,"B18,B22,B25,B26,B28,B29,B32,B33,B35,B36,B39,B4...",1.15,...,1.47,1.44,NaN,NaN,NaN,"0 hours, 4 minutes, 2 seconds",NaN,IgG_Fc_5JII_AB,relaxed_filters,default_4stage_multimer


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, glob, shutil
import pandas as pd
from google.colab import files

# Change this to your current BindCraft output folder if you know it
BASE = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"

# If your Drive uses "My Drive" instead of "MyDrive", use this instead:
# BASE = "/content/drive/My Drive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"

print("Base exists:", os.path.exists(BASE))

csv_files = glob.glob(os.path.join(BASE, "**", "*.csv"), recursive=True)

print("CSV files found:", len(csv_files))
for f in csv_files:
    print(f)

# Make a local folder for export
export_dir = "/content/BindCraft_CSV_Export"
os.makedirs(export_dir, exist_ok=True)

# Copy all CSVs into export folder
for f in csv_files:
    new_name = os.path.basename(f)
    shutil.copy(f, os.path.join(export_dir, new_name))

# Zip all CSV files
zip_path = "/content/BindCraft_CSV_Export.zip"
shutil.make_archive("/content/BindCraft_CSV_Export", "zip", export_dir)

print("Created zip:", zip_path)

# Download zip to your computer
files.download(zip_path)

Base exists: True
CSV files found: 4
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/mpnn_design_stats.csv
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/final_design_stats.csv
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/trajectory_stats.csv
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/failure_csv.csv
Created zip: /content/BindCraft_CSV_Export.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title Corrected DEFAULT filter retest for accepted BindCraft binders

import os, glob, json, ast
import pandas as pd
from IPython.display import display

# ============================================================
# Your actual BindCraft run folder from the screenshot
# ============================================================
RUN_DIR = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"

# Optional: test only one binder by exact name.
# Leave as None to test all accepted/final binders.
BINDER_NAME = None
# Example:
# BINDER_NAME = "IgG_Fc_5JII_AB_l72_s140142_mpnn1"


# ============================================================
# Basic checks
# ============================================================
print("Run folder exists:", os.path.exists(RUN_DIR))
if not os.path.exists(RUN_DIR):
    raise FileNotFoundError(f"RUN_DIR not found: {RUN_DIR}")

final_csvs = glob.glob(os.path.join(RUN_DIR, "**", "final_design_stats.csv"), recursive=True)
mpnn_csvs = glob.glob(os.path.join(RUN_DIR, "**", "mpnn_design_stats.csv"), recursive=True)

print("\nfinal_design_stats.csv found:")
for f in final_csvs:
    print(" ", f)

print("\nmpnn_design_stats.csv found:")
for f in mpnn_csvs:
    print(" ", f)

if not final_csvs and not mpnn_csvs:
    raise FileNotFoundError("No final_design_stats.csv or mpnn_design_stats.csv found in this run folder.")

# Prefer accepted/final designs
csv_path = final_csvs[0] if final_csvs else mpnn_csvs[0]
print("\nUsing CSV:")
print(csv_path)


# ============================================================
# Confirm BindCraft functions/labels are loaded
# ============================================================
try:
    check_filters
except NameError:
    raise NameError("Run the BindCraft cell 'Import functions and settings' first. check_filters is not loaded.")

try:
    design_labels
except NameError:
    raise NameError("Run the BindCraft cell 'Import functions and settings' first. design_labels is not loaded.")


# ============================================================
# Load accepted/final design CSV
# ============================================================
df_raw = pd.read_csv(csv_path)

print("\nRaw CSV shape:", df_raw.shape)
print("Number of BindCraft design_labels:", len(design_labels))

# final_design_stats.csv often has one extra ranking/index column
if df_raw.shape[1] == len(design_labels) + 1:
    df = df_raw.iloc[:, 1:].copy()
    df.columns = design_labels
elif df_raw.shape[1] == len(design_labels):
    df = df_raw.copy()
    df.columns = design_labels
else:
    print("\nCSV columns:")
    print(list(df_raw.columns))
    raise ValueError(
        f"Column mismatch: CSV has {df_raw.shape[1]} columns, design_labels has {len(design_labels)}."
    )


# ============================================================
# Convert dictionary-like CSV strings back to Python dicts
# This fixes the error: 'str' object has no attribute 'get'
# ============================================================
def parse_dict_if_needed(x):
    if isinstance(x, dict):
        return x

    if pd.isna(x):
        return None

    if isinstance(x, str):
        s = x.strip()
        if s.startswith("{") and s.endswith("}"):
            try:
                return ast.literal_eval(s)
            except Exception:
                try:
                    return json.loads(s.replace("'", '"'))
                except Exception:
                    return x

    return x

for col in df.columns:
    if "InterfaceAAs" in col:
        df[col] = df[col].apply(parse_dict_if_needed)

print("\nConverted InterfaceAAs columns:")
print([c for c in df.columns if "InterfaceAAs" in c])


# ============================================================
# Find the DEFAULT filter JSON
# ============================================================
# First, use the Filters path inside the CSV if available.
filter_candidates = []

if "Filters" in df.columns:
    for p in df["Filters"].dropna().unique():
        p = str(p)
        if os.path.exists(p):
            filter_dir = os.path.dirname(p)
            filter_candidates.extend(glob.glob(os.path.join(filter_dir, "*default*.json")))
            filter_candidates.extend(glob.glob(os.path.join(filter_dir, "*Default*.json")))

# Also search common BindCraft locations.
for root in ["/content/bindcraft", "/content/BindCraft", "/content"]:
    if os.path.exists(root):
        filter_candidates.extend(glob.glob(os.path.join(root, "**", "*default*.json"), recursive=True))
        filter_candidates.extend(glob.glob(os.path.join(root, "**", "*Default*.json"), recursive=True))

filter_candidates = sorted(set(filter_candidates))

print("\nDefault-filter JSON candidates:")
for p in filter_candidates:
    print(" ", p)

def load_possible_filter_json(path):
    with open(path, "r") as f:
        data = json.load(f)

    # Some JSON files may wrap the filter dictionary
    if isinstance(data, dict):
        if any(k in data for k in ["Average_pLDDT", "Average_i_pTM", "Average_i_pAE", "Average_Binder_RMSD"]):
            return data

        for key in ["filters", "Filters", "default", "Default"]:
            if key in data and isinstance(data[key], dict):
                inner = data[key]
                if any(k in inner for k in ["Average_pLDDT", "Average_i_pTM", "Average_i_pAE", "Average_Binder_RMSD"]):
                    return inner

    return None

default_filters = None
DEFAULT_FILTER_JSON = None

for candidate in filter_candidates:
    filt = load_possible_filter_json(candidate)
    if filt is not None:
        default_filters = filt
        DEFAULT_FILTER_JSON = candidate
        break

if default_filters is None:
    raise FileNotFoundError(
        "Could not identify the Default filter JSON. "
        "Try selecting 'Default' in the BindCraft Filters dropdown and run the Filters cell once."
    )

print("\nUsing DEFAULT filter file:")
print(DEFAULT_FILTER_JSON)

print("\nDefault filter keys preview:")
print(list(default_filters.keys())[:20])


# ============================================================
# Optional: test only one binder
# ============================================================
if BINDER_NAME is not None:
    df = df[df["Design"].astype(str) == BINDER_NAME].copy()
    if df.empty:
        raise ValueError(f"Binder not found: {BINDER_NAME}")

print("\nDesigns to test:", len(df))

preview_cols = [
    "Design",
    "Length",
    "Average_pLDDT",
    "Average_i_pTM",
    "Average_i_pAE",
    "Average_Binder_RMSD",
    "Average_ShapeComplementarity",
    "Average_dG",
    "Average_Relaxed_Clashes"
]

available_preview_cols = [c for c in preview_cols if c in df.columns]
display(df[available_preview_cols])


# ============================================================
# Run DEFAULT filter check
# ============================================================
results = []

for _, row in df.iterrows():

    mpnn_data = []

    for label in design_labels:
        value = row[label] if label in row.index else None

        # Convert NaN to None, but do not break dict values
        if not isinstance(value, dict):
            try:
                if pd.isna(value):
                    value = None
            except Exception:
                pass

        mpnn_data.append(value)

    try:
        failed = check_filters(mpnn_data, design_labels, default_filters)

        if failed is True:
            passed_default = True
            failed_metrics = ""
        else:
            passed_default = False
            failed_metrics = ", ".join([str(x) for x in failed])

    except Exception as e:
        passed_default = False
        failed_metrics = f"ERROR during check_filters: {repr(e)}"

    results.append({
        "Design": row.get("Design", None),
        "Relaxed_filter_accepted": True,
        "Default_filter_pass": passed_default,
        "Failed_default_filter_metrics": failed_metrics,
        "Average_pLDDT": row.get("Average_pLDDT", None),
        "Average_i_pTM": row.get("Average_i_pTM", None),
        "Average_i_pAE": row.get("Average_i_pAE", None),
        "Average_Binder_RMSD": row.get("Average_Binder_RMSD", None),
        "Average_ShapeComplementarity": row.get("Average_ShapeComplementarity", None),
        "Average_dG": row.get("Average_dG", None),
        "Average_Relaxed_Clashes": row.get("Average_Relaxed_Clashes", None),
        "Default_filter_file_used": DEFAULT_FILTER_JSON
    })

results_df = pd.DataFrame(results)

print("\nDEFAULT FILTER RETEST RESULTS")
display(results_df)


# ============================================================
# Save result as CSV
# ============================================================
out_csv = os.path.join(RUN_DIR, "default_filter_retest_results_corrected.csv")
results_df.to_csv(out_csv, index=False)

print("\nSaved corrected default-filter retest CSV here:")
print(out_csv)

print("\nSummary:")
print(results_df["Default_filter_pass"].value_counts(dropna=False))

Run folder exists: True

final_design_stats.csv found:
  /content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/final_design_stats.csv

mpnn_design_stats.csv found:
  /content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/mpnn_design_stats.csv

Using CSV:
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/final_design_stats.csv

Raw CSV shape: (3, 232)
Number of BindCraft design_labels: 231

Converted InterfaceAAs columns:
['Average_InterfaceAAs', '1_InterfaceAAs', '2_InterfaceAAs', '3_InterfaceAAs', '4_InterfaceAAs', '5_InterfaceAAs']

Default-filter JSON candidates:
  /content/bindcraft/settings_advanced/default_4stage_multimer.json
  /content/bindcraft/settings_advanced/default_4stage_multimer_flexible.json
  /content/bindcraft/settings_advanced/default_4stage_multimer_flexible_hardtarget.json
  /content/bindcraft/settings_advanced/default_4stage_multimer_hardtarget.json
  /content/bindcraft/settings_advanced/default_4stage_multimer_mpnn.json
  /content/bindcraft/set

,Design,Length,Average_pLDDT,Average_i_pTM,Average_i_pAE,Average_Binder_RMSD,Average_ShapeComplementarity,Average_dG,Average_Relaxed_Clashes
0,IgG_Fc_5JII_AB_l72_s140142_mpnn1,72,0.90,0.80,0.29,1.28,0.69,-41.91,0.0
1,IgG_Fc_5JII_AB_l72_s140142_mpnn4,72,0.89,0.78,0.30,1.36,0.71,-43.94,0.0
2,IgG_Fc_5JII_AB_l63_s343113_mpnn1,63,0.86,0.69,0.37,1.46,0.66,-55.79,0.0



DEFAULT FILTER RETEST RESULTS


,Design,Relaxed_filter_accepted,Default_filter_pass,Failed_default_filter_metrics,Average_pLDDT,Average_i_pTM,Average_i_pAE,Average_Binder_RMSD,Average_ShapeComplementarity,Average_dG,Average_Relaxed_Clashes,Default_filter_file_used
0,IgG_Fc_5JII_AB_l72_s140142_mpnn1,True,False,"1_n_InterfaceHbonds, Average_InterfaceAAs_M",0.90,0.80,0.29,1.28,0.69,-41.91,0.0,/content/bindcraft/settings_filters/default_fi...
1,IgG_Fc_5JII_AB_l72_s140142_mpnn4,True,False,Average_InterfaceAAs_M,0.89,0.78,0.30,1.36,0.71,-43.94,0.0,/content/bindcraft/settings_filters/default_fi...
2,IgG_Fc_5JII_AB_l63_s343113_mpnn1,True,False,"Average_i_pAE, 1_i_pAE, 2_i_pAE, Average_Inter...",0.86,0.69,0.37,1.46,0.66,-55.79,0.0,/content/bindcraft/settings_filters/default_fi...



Saved corrected default-filter retest CSV here:
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/default_filter_retest_results_corrected.csv

Summary:
Default_filter_pass
False    3
Name: count, dtype: int64


In [ ]:
#@title Generate Excel report: Default filter retest for accepted binders

import os, glob, json, ast
import pandas as pd
from IPython.display import display
from google.colab import files

# ============================================================
# Your BindCraft run folder
# ============================================================
RUN_DIR = "/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/"

# Optional: test only one binder by exact name.
# Leave as None to test all accepted binders.
BINDER_NAME = None
# Example:
# BINDER_NAME = "IgG_Fc_5JII_AB_l72_s140142_mpnn1"

OUTPUT_XLSX = os.path.join(RUN_DIR, "default_filter_retest_results.xlsx")

print("Run folder exists:", os.path.exists(RUN_DIR))
if not os.path.exists(RUN_DIR):
    raise FileNotFoundError(f"RUN_DIR not found: {RUN_DIR}")


# ============================================================
# Find accepted/final design CSV
# ============================================================
final_csvs = glob.glob(os.path.join(RUN_DIR, "**", "final_design_stats.csv"), recursive=True)
mpnn_csvs = glob.glob(os.path.join(RUN_DIR, "**", "mpnn_design_stats.csv"), recursive=True)

if not final_csvs and not mpnn_csvs:
    raise FileNotFoundError("No final_design_stats.csv or mpnn_design_stats.csv found.")

csv_path = final_csvs[0] if final_csvs else mpnn_csvs[0]

print("Using design CSV:")
print(csv_path)


# ============================================================
# Confirm BindCraft variables/functions are available
# ============================================================
try:
    check_filters
except NameError:
    raise NameError("Please run the BindCraft 'Import functions and settings' cell first.")

try:
    design_labels
except NameError:
    raise NameError("Please run the BindCraft 'Import functions and settings' cell first.")


# ============================================================
# Load accepted binder data
# ============================================================
df_raw = pd.read_csv(csv_path)

if df_raw.shape[1] == len(design_labels) + 1:
    df = df_raw.iloc[:, 1:].copy()
    df.columns = design_labels
elif df_raw.shape[1] == len(design_labels):
    df = df_raw.copy()
    df.columns = design_labels
else:
    raise ValueError(
        f"Column mismatch: CSV has {df_raw.shape[1]} columns, design_labels has {len(design_labels)}."
    )

if BINDER_NAME is not None:
    df = df[df["Design"].astype(str) == BINDER_NAME].copy()
    if df.empty:
        raise ValueError(f"Binder not found: {BINDER_NAME}")

print("Accepted binders loaded:", len(df))


# ============================================================
# Convert InterfaceAAs text back to dictionary
# Fixes: 'str' object has no attribute 'get'
# ============================================================
def parse_dict_if_needed(x):
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("{") and s.endswith("}"):
            try:
                return ast.literal_eval(s)
            except Exception:
                return x
    return x

for col in df.columns:
    if "InterfaceAAs" in col:
        df[col] = df[col].apply(parse_dict_if_needed)


# ============================================================
# Find Default filter JSON
# ============================================================
filter_candidates = []

if "Filters" in df.columns:
    for p in df["Filters"].dropna().unique():
        p = str(p)
        if os.path.exists(p):
            filter_dir = os.path.dirname(p)
            filter_candidates.extend(glob.glob(os.path.join(filter_dir, "*default*.json")))
            filter_candidates.extend(glob.glob(os.path.join(filter_dir, "*Default*.json")))

for root in ["/content/bindcraft", "/content/BindCraft", "/content"]:
    if os.path.exists(root):
        filter_candidates.extend(glob.glob(os.path.join(root, "**", "*default*.json"), recursive=True))
        filter_candidates.extend(glob.glob(os.path.join(root, "**", "*Default*.json"), recursive=True))

filter_candidates = sorted(set(filter_candidates))

def load_possible_filter_json(path):
    with open(path, "r") as f:
        data = json.load(f)

    if isinstance(data, dict):
        if any(k in data for k in ["Average_pLDDT", "Average_i_pTM", "Average_i_pAE", "Average_Binder_RMSD"]):
            return data

        for key in ["filters", "Filters", "default", "Default"]:
            if key in data and isinstance(data[key], dict):
                inner = data[key]
                if any(k in inner for k in ["Average_pLDDT", "Average_i_pTM", "Average_i_pAE", "Average_Binder_RMSD"]):
                    return inner

    return None

default_filters = None
DEFAULT_FILTER_JSON = None

for candidate in filter_candidates:
    filt = load_possible_filter_json(candidate)
    if filt is not None:
        default_filters = filt
        DEFAULT_FILTER_JSON = candidate
        break

if default_filters is None:
    raise FileNotFoundError(
        "Default filter JSON not found. Select 'Default' in the BindCraft Filters dropdown and run the Filters cell once."
    )

print("Using Default filter file:")
print(DEFAULT_FILTER_JSON)


# ============================================================
# Run Default filter retest
# ============================================================
results = []

for _, row in df.iterrows():

    mpnn_data = []

    for label in design_labels:
        value = row[label] if label in row.index else None

        if not isinstance(value, dict):
            try:
                if pd.isna(value):
                    value = None
            except Exception:
                pass

        mpnn_data.append(value)

    try:
        failed = check_filters(mpnn_data, design_labels, default_filters)

        if failed is True:
            passed_default = True
            failed_metrics = ""
        else:
            passed_default = False
            failed_metrics = ", ".join([str(x) for x in failed])

    except Exception as e:
        passed_default = False
        failed_metrics = f"ERROR during check_filters: {repr(e)}"

    results.append({
        "Design": row.get("Design", None),
        "Length": row.get("Length", None),
        "Relaxed_filter_accepted": True,
        "Default_filter_pass": passed_default,
        "Failed_default_filter_metrics": failed_metrics,
        "Average_pLDDT": row.get("Average_pLDDT", None),
        "Average_i_pTM": row.get("Average_i_pTM", None),
        "Average_i_pAE": row.get("Average_i_pAE", None),
        "Average_Binder_pLDDT": row.get("Average_Binder_pLDDT", None),
        "Average_Binder_RMSD": row.get("Average_Binder_RMSD", None),
        "Average_ShapeComplementarity": row.get("Average_ShapeComplementarity", None),
        "Average_dG": row.get("Average_dG", None),
        "Average_dSASA": row.get("Average_dSASA", None),
        "Average_n_InterfaceResidues": row.get("Average_n_InterfaceResidues", None),
        "Average_n_InterfaceHbonds": row.get("Average_n_InterfaceHbonds", None),
        "Average_Relaxed_Clashes": row.get("Average_Relaxed_Clashes", None),
        "Default_filter_file_used": DEFAULT_FILTER_JSON
    })

results_df = pd.DataFrame(results)

print("\nDefault filter retest result:")
display(results_df)


# ============================================================
# Make summary sheet
# ============================================================
summary_df = pd.DataFrame({
    "Metric": [
        "Run folder",
        "Input CSV used",
        "Default filter file used",
        "Total relaxed-filter accepted binders tested",
        "Number passing Default filter",
        "Number failing Default filter"
    ],
    "Value": [
        RUN_DIR,
        csv_path,
        DEFAULT_FILTER_JSON,
        len(results_df),
        int(results_df["Default_filter_pass"].sum()),
        int((~results_df["Default_filter_pass"]).sum())
    ]
})


# ============================================================
# Save Excel file with multiple sheets
# ============================================================
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    results_df.to_excel(writer, sheet_name="Default_Filter_Retest", index=False)
    df.to_excel(writer, sheet_name="Accepted_Binder_Raw_Data", index=False)

    # Basic formatting
    workbook = writer.book

    for sheet_name in workbook.sheetnames:
        ws = workbook[sheet_name]

        # Freeze top row
        ws.freeze_panes = "A2"

        # Auto-width columns
        for column_cells in ws.columns:
            max_length = 0
            column_letter = column_cells[0].column_letter

            for cell in column_cells:
                try:
                    value = str(cell.value) if cell.value is not None else ""
                    max_length = max(max_length, len(value))
                except Exception:
                    pass

            adjusted_width = min(max(max_length + 2, 12), 45)
            ws.column_dimensions[column_letter].width = adjusted_width

        # Header formatting
        for cell in ws[1]:
            cell.font = cell.font.copy(bold=True)
            cell.alignment = cell.alignment.copy(wrap_text=True)

print("\nExcel file saved here:")
print(OUTPUT_XLSX)

# Download to your computer
files.download(OUTPUT_XLSX)

Run folder exists: True
Using design CSV:
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/final_design_stats.csv
Accepted binders loaded: 3
Using Default filter file:
/content/bindcraft/settings_filters/default_filters.json

Default filter retest result:


,Design,Length,Relaxed_filter_accepted,Default_filter_pass,Failed_default_filter_metrics,Average_pLDDT,Average_i_pTM,Average_i_pAE,Average_Binder_pLDDT,Average_Binder_RMSD,Average_ShapeComplementarity,Average_dG,Average_dSASA,Average_n_InterfaceResidues,Average_n_InterfaceHbonds,Average_Relaxed_Clashes,Default_filter_file_used
0,IgG_Fc_5JII_AB_l72_s140142_mpnn1,72,True,False,"1_n_InterfaceHbonds, Average_InterfaceAAs_M",0.90,0.80,0.29,0.91,1.28,0.69,-41.91,1612.68,17.5,3.0,0.0,/content/bindcraft/settings_filters/default_fi...
1,IgG_Fc_5JII_AB_l72_s140142_mpnn4,72,True,False,Average_InterfaceAAs_M,0.89,0.78,0.30,0.91,1.36,0.71,-43.94,1576.32,18.5,4.0,0.0,/content/bindcraft/settings_filters/default_fi...
2,IgG_Fc_5JII_AB_l63_s343113_mpnn1,63,True,False,"Average_i_pAE, 1_i_pAE, 2_i_pAE, Average_Inter...",0.86,0.69,0.37,0.92,1.46,0.66,-55.79,2038.84,21.0,6.0,0.0,/content/bindcraft/settings_filters/default_fi...



Excel file saved here:
/content/drive/MyDrive/BindCraft/IgG_Fc_5JII_CLEAN/AB_run1/default_filter_retest_results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title Consolidate & Rank Designs
#@markdown ---
accepted_binders = [f for f in os.listdir(design_paths["Accepted"]) if f.endswith('.pdb')]

for f in os.listdir(design_paths["Accepted/Ranked"]):
    os.remove(os.path.join(design_paths["Accepted/Ranked"], f))

# load dataframe of designed binders
design_df = pd.read_csv(mpnn_csv)
design_df = design_df.sort_values('Average_i_pTM', ascending=False)

# create final csv dataframe to copy matched rows, initialize with the column labels
final_df = pd.DataFrame(columns=final_labels)

# check the ranking of the designs and copy them with new ranked IDs to the folder
rank = 1
for _, row in design_df.iterrows():
    for binder in accepted_binders:
        target_settings["binder_name"], model = binder.rsplit('_model', 1)
        if target_settings["binder_name"] == row['Design']:
            # rank and copy into ranked folder
            row_data = {'Rank': rank, **{label: row[label] for label in design_labels}}
            final_df = pd.concat([final_df, pd.DataFrame([row_data])], ignore_index=True)
            old_path = os.path.join(design_paths["Accepted"], binder)
            new_path = os.path.join(design_paths["Accepted/Ranked"], f"{rank}_{target_settings['binder_name']}_model{model.rsplit('.', 1)[0]}.pdb")
            shutil.copyfile(old_path, new_path)

            rank += 1
            break

# save the final_df to final_csv
final_df.to_csv(final_csv, index=False)

print("Designs ranked and final_designs_stats.csv generated")

In [ ]:
#@title Top 20 Designs
df = pd.read_csv(os.path.join(design_path, 'final_design_stats.csv'))
df.head(20)

In [ ]:
#@title Top Design Display
import py3Dmol
import glob
from IPython.display import HTML

#### pymol top design
top_design_dir = os.path.join(design_path, 'Accepted', 'Ranked')
top_design_pdb = glob.glob(os.path.join(top_design_dir, '1_*.pdb'))[0]

# Visualise in PyMOL
view = py3Dmol.view()
view.addModel(open(top_design_pdb, 'r').read(),'pdb')
view.setBackgroundColor('white')
view.setStyle({'chain':'A'}, {'cartoon': {'color':'#3c5b6f'}})
view.setStyle({'chain':'B'}, {'cartoon': {'color':'#B76E79'}})
view.zoomTo()
view.show()

In [ ]:
#@title Display animation
import glob
from IPython.display import HTML

#### pymol top design
top_design_dir = os.path.join(design_path, 'Accepted', 'Ranked')
top_design_pdb = glob.glob(os.path.join(top_design_dir, '1_*.pdb'))[0]

top_design_name = os.path.basename(top_design_pdb).split('1_', 1)[1].split('_mpnn')[0]
top_design_animation = os.path.join(design_path, 'Accepted', 'Animation', f"{top_design_name}.html")

# Show animation
HTML(top_design_animation)